# Vehicle Type Classifier Training
## Rock Cluster Camera Brain Project

Trains MobileNetV3-Small to classify vehicle types:
- sedan, SUV, truck, van, pickup, bus, motorcycle, bicycle

**GPU Required:** Runtime → Change runtime type → GPU (T4)

In [ ]:
# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB" if torch.cuda.is_available() else "N/A")

In [ ]:
# Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install albumentations pandas tqdm pillow matplotlib
!pip install onnx onnxruntime

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from torchvision.models import MobileNet_V3_Small_Weights
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pandas as pd
import numpy as np
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
import onnx

## Dataset Setup

Choose one option below:

In [ ]:
# OPTION 1: Download Stanford Cars (automatic)
# Note: Stanford Cars requires manual download approval
# For demo, we'll create synthetic data structure

# Create directory structure
!mkdir -p /content/data/train /content/data/val

VEHICLE_CLASSES = ['sedan', 'SUV', 'truck', 'van', 'pickup', 'bus', 'motorcycle', 'bicycle']
for cls in VEHICLE_CLASSES:
    !mkdir -p /content/data/train/{cls}
    !mkdir -p /content/data/val/{cls}

print(f"Created directories for {len(VEHICLE_CLASSES)} classes: {VEHICLE_CLASSES}")

# Upload your dataset using the file uploader on the left panel
# Or mount Google Drive:
from google.colab import drive
# drive.mount('/content/drive')  # Uncomment to use Drive

print("\nUpload your dataset to /content/data/train and /content/data/val")
print("Format: /content/data/train/{class_name}/{image}.jpg")

In [ ]:
# Dataset class
class VehicleDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.classes = VEHICLE_CLASSES
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        self.samples = []
        for cls in self.classes:
            cls_dir = self.root_dir / cls
            if cls_dir.exists():
                for img_path in cls_dir.glob('*.jpg'):
                    self.samples.append((str(img_path), self.class_to_idx[cls]))
                for img_path in cls_dir.glob('*.png'):
                    self.samples.append((str(img_path), self.class_to_idx[cls]))
        
        print(f"Loaded {len(self.samples)} images from {root_dir}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = np.array(Image.open(img_path).convert('RGB'))
        
        if self.transform:
            img = self.transform(image=img)['image']
        
        return img, label

# Data augmentation
train_transform = A.Compose([
    A.Resize(256, 256),
    A.RandomCrop(224, 224),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    A.RandomRotate90(p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

In [ ]:
# Create model
def create_vehicle_classifier(num_classes=8, pretrained=True):
    model = models.mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None)
    
    # Replace classifier head
    num_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(num_features, num_classes)
    
    return model

NUM_CLASSES = len(VEHICLE_CLASSES)
model = create_vehicle_classifier(NUM_CLASSES)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Model created: MobileNetV3-Small with {NUM_CLASSES} output classes")
print(f"Device: {device}")

In [ ]:
# Create data loaders
BATCH_SIZE = 32
NUM_WORKERS = 2

train_dataset = VehicleDataset('/content/data/train', transform=train_transform)
val_dataset = VehicleDataset('/content/data/val', transform=val_transform)

print(f"\nTraining samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

if len(train_dataset) == 0:
    print("\n⚠️ WARNING: No training data found!")
    print("Please upload images to /content/data/train/{class_name}/")
    print(f"Expected classes: {VEHICLE_CLASSES}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(loader, desc='Training'):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    
    return running_loss / len(loader), correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validating'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    
    return running_loss / len(loader), correct / total

In [ ]:
# Training loop
NUM_EPOCHS = 30
best_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print(f"Starting training for {NUM_EPOCHS} epochs...")
print(f"Classes: {VEHICLE_CLASSES}")

for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'classes': VEHICLE_CLASSES
        }, '/content/vehicle_classifier_best.pth')
        print(f"✓ Best model saved! Val Acc: {best_acc:.4f}")

print(f"\n✓ Training complete! Best validation accuracy: {best_acc:.4f}")

In [ ]:
# Plot training curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss Curves')

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy Curves')

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()

In [ ]:
# Export to ONNX
print("Exporting model to ONNX...")

model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)

# Export
torch.onnx.export(
    model,
    dummy_input,
    '/content/vehicle_classifier.onnx',
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    opset_version=11
)

# Verify ONNX model
onnx_model = onnx.load('/content/vehicle_classifier.onnx')
onnx.checker.check_model(onnx_model)
print("✓ ONNX model exported and verified!")

# Save class mapping
config = {
    'classes': VEHICLE_CLASSES,
    'input_size': [224, 224],
    'model_type': 'mobilenet_v3_small',
    'num_classes': NUM_CLASSES
}

with open('/content/vehicle_classifier_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✓ Config saved!")

In [ ]:
# Download files
from google.colab import files

print("Downloading trained models...")
files.download('/content/vehicle_classifier_best.pth')
files.download('/content/vehicle_classifier.onnx')
files.download('/content/vehicle_classifier_config.json')
files.download('/content/training_curves.png')

print("\n✓ All files downloaded!")
print("\nNext step: Convert ONNX to RKNN for RK3568 NPU deployment")
print("See: /home/camera-brain/training/rknn_conversion.py")